In [25]:
from pathlib import Path

raw_path = Path("data/the-verdict.txt")

with open(raw_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


# Simple Tokenizer

In [34]:
# Construct Vocabulary

import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
#preprocessed = [stripped for item in preprocessed if (stripped := item.strip())]

all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<endoftext>", "<|unk|>"])
vocab = {token: i for i, token in enumerate(all_tokens)}

# -----------------------------------
print("-- Processed Text --")
print(len(preprocessed))
print(preprocessed[:30])
print(" ")

print("-- Vocabulary --")
print(len(vocab.items()))
for i , item in enumerate(vocab.items()):
    print(item)
    if i>=20:
        break

-- Processed Text --
9235
['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ', 'Gisburn', ' ', 'rather', ' ', 'a', ' ', 'cheap', ' ', 'genius', '--', 'though', ' ', 'a', ' ', 'good', ' ', 'fellow', ' ', 'enough', '--']
 
-- Vocabulary --
1135
('', 0)
('\n', 1)
(' ', 2)
('!', 3)
('"', 4)
("'", 5)
('(', 6)
(')', 7)
(',', 8)
('--', 9)
('.', 10)
(':', 11)
(';', 12)
('?', 13)
('A', 14)
('Ah', 15)
('Among', 16)
('And', 17)
('Are', 18)
('Arrt', 19)
('As', 20)


In [43]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [stripped for item in preprocessed if (stripped:=item.strip())]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [44]:
text1 = "Hello, do you know tea?"
text2 = "In the sunlit terrances of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

tokenizer = SimpleTokenizer(vocab)

print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

Hello, do you know tea? <|endoftext|> In the sunlit terrances of the palace.
[1134, 8, 358, 1129, 599, 978, 13, 1134, 58, 991, 959, 1134, 725, 991, 1134, 10]
<|unk|>, do you know tea? <|unk|> In the sunlit <|unk|> of the <|unk|>.


# BPE Tokenizer

In [45]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [47]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace"
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271]


In [48]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace


In [62]:
# Excercise 2.1

unknown_str = "Akwirw ier"
integers = tokenizer.encode(unknown_str)
strings = [tokenizer.decode([i]) for i in integers]
text = tokenizer.decode(integers)

print(integers)
print(strings)
print(text)


[33901, 86, 343, 86, 220, 959]
['Ak', 'w', 'ir', 'w', ' ', 'ier']
Akwirw ier


# Dataset Construction